# Semana 1: Estructuras de Datos para Búsqueda en Bases de Datos (SQL y NoSQL)
## Notebook Conceptual (NB1) – Algoritmos y Estructuras: El "Backend" de las Bases de Datos

### Propósito de la sesión
Comprender que **las bases de datos no son cajas mágicas, sino implementaciones sofisticadas de estructuras de datos clásicas**. Mapearemos estas estructuras (árboles, tablas hash, listas, skip lists, LSM-Trees) con motores específicos (PostgreSQL, MySQL, MongoDB, Redis, Cassandra, Neo4j, Elasticsearch) y analizaremos cómo determinan la complejidad algorítmica de las operaciones y, por tanto, el rendimiento.

### Objetivos de aprendizaje
*   Entender el funcionamiento de un **B-Tree** y su relación con índices SQL.
*   Implementar una **tabla hash** y medir su eficiencia O(1).
*   Simular una **lista enlazada** y comprender su uso en colas.
*   Conocer las **Skip Lists** y su implementación en Redis (Sorted Sets).
*   Simular un **LSM-Tree** y entender su optimización para escrituras.
*   Construir un **índice invertido** para búsqueda de texto.
*   Representar un **grafo** y relacionarlo con Neo4j.
*   Completar una tabla resumen mapeando motores con sus estructuras subyacentes.

## Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno. Incluimos enlaces a visualizadores web interactivos.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from collections import defaultdict
import hashlib

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")
print("\n🔗 Enlaces a visualizadores interactivos:")
print("  - B-Tree Visualizer: https://www.cs.usfca.edu/~galles/visualization/BTree.html")
print("  - Skip List Visualizer: https://www.cs.usfca.edu/~galles/visualization/SkipList.html")

---
## Actividad 1: Fundamento - El B-Tree y los Índices SQL

Los B-Trees (árboles balanceados) son la estructura de índice por defecto en la mayoría de las bases de datos relacionales (PostgreSQL, MySQL, Oracle, SQL Server) y también en MongoDB.

### 1.1. Visualización interactiva

**Instrucciones:**
1. Abre el [B-Tree Visualizer de la USFCA](https://www.cs.usfca.edu/~galles/visualization/BTree.html).
2. Selecciona un orden (por ejemplo, 4) y haz clic en "Insert".
3. Inserta las claves: 10, 20, 5, 6, 12, 30, 15, 17.
4. Observa cómo el árbol se divide cuando un nodo se llena.
5. Realiza una búsqueda (por ejemplo, 12) y observa el camino desde la raíz hasta la hoja.

### 1.2. Relación con la complejidad O(log n)

La profundidad de un B-Tree es logarítmica respecto al número de claves. El factor de ramificación (fanout) reduce la altura. Por ejemplo, con un fanout de 500, un árbol puede manejar millones de claves con solo 3-4 niveles.

In [ ]:
# Simulación conceptual: cálculo de niveles necesarios
def btree_levels(n, fanout):
    """Calcula la altura aproximada de un B-Tree."""
    levels = 0
    total_keys = 1
    while total_keys < n:
        total_keys *= fanout
        levels += 1
    return levels

n_keys = 10_000_000  # 10 millones de claves
fanout = 500
height = btree_levels(n_keys, fanout)
print(f"Para {n_keys:,} claves con fanout {fanout}, altura ≈ {height}")
print(f"Una búsqueda requiere ~{height} accesos a nodos (cada acceso puede ser un I/O).")
print(f"✅ Complejidad O(log n) con base alta.")

---
## Actividad 2: Búsqueda por Hash - La Estructura Detrás de las Claves-Valor

Las tablas hash permiten búsqueda, inserción y eliminación en **tiempo constante O(1)** en promedio.

### 2.1. Implementación de una tabla hash simple con encadenamiento

In [ ]:
class HashTable:
    def __init__(self, size=10):
        self.size = size
        self.table = [[] for _ in range(size)]
        self.num_elements = 0
    
    def _hash(self, key):
        """Función hash simple."""
        return hash(key) % self.size
    
    def insert(self, key, value):
        idx = self._hash(key)
        # Verificar si la clave ya existe para actualizar
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx][i] = (key, value)
                return
        # Si no existe, agregar nueva
        self.table[idx].append((key, value))
        self.num_elements += 1
    
    def search(self, key):
        idx = self._hash(key)
        for k, v in self.table[idx]:
            if k == key:
                return v
        return None
    
    def delete(self, key):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx].pop(i)
                self.num_elements -= 1
                return True
        return False
    
    def load_factor(self):
        return self.num_elements / self.size

# Probamos la tabla hash
ht = HashTable(size=5)
ht.insert("clave1", 100)
ht.insert("clave2", 200)
ht.insert("clave3", 300)
ht.insert("clave1", 150)  # actualización

print(f"Búsqueda 'clave1': {ht.search('clave1')}")
print(f"Búsqueda 'clave4': {ht.search('clave4')}")
print(f"Factor de carga: {ht.load_factor():.2f}")

# Medimos tiempos
keys = [f"key_{i}" for i in range(1000)]
values = list(range(1000))

ht_big = HashTable(size=1000)
start = time.time()
for k, v in zip(keys, values):
    ht_big.insert(k, v)
t_insert = time.time() - start

start = time.time()
for k in keys:
    ht_big.search(k)
t_search = time.time() - start

print(f"\nTiempo para insertar 1000 elementos: {t_insert:.6f} s")
print(f"Tiempo para buscar 1000 elementos: {t_search:.6f} s")
print(f"Tiempo promedio por operación: {(t_insert+t_search)/2000:.2e} s")

### 2.2. Conexión con motores

*   **Redis**: Es esencialmente una tabla hash gigante en memoria que almacena claves que apuntan a diversas estructuras de datos.
*   **PostgreSQL**: Soporta índices hash para búsquedas de igualdad exacta (`CREATE INDEX ... USING HASH`).
*   **MongoDB**: Ofrece índices hashed, principalmente para distribuir datos en sharding.

---
## Actividad 3: Listas Enlazadas y el Corazón de las Colas

Las listas doblemente enlazadas permiten inserciones y eliminaciones en ambos extremos en O(1).

### 3.1. Implementación de una lista doblemente enlazada

In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.prev = None
        self.next = None

class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0
    
    def append_left(self, data):
        new_node = Node(data)
        if self.head is None:
            self.head = self.tail = new_node
        else:
            new_node.next = self.head
            self.head.prev = new_node
            self.head = new_node
        self.size += 1
    
    def append_right(self, data):
        new_node = Node(data)
        if self.tail is None:
            self.head = self.tail = new_node
        else:
            new_node.prev = self.tail
            self.tail.next = new_node
            self.tail = new_node
        self.size += 1
    
    def pop_left(self):
        if self.head is None:
            return None
        data = self.head.data
        self.head = self.head.next
        if self.head:
            self.head.prev = None
        else:
            self.tail = None
        self.size -= 1
        return data
    
    def pop_right(self):
        if self.tail is None:
            return None
        data = self.tail.data
        self.tail = self.tail.prev
        if self.tail:
            self.tail.next = None
        else:
            self.head = None
        self.size -= 1
        return data
    
    def get_at_index(self, index):
        if index < 0 or index >= self.size:
            return None
        current = self.head
        for _ in range(index):
            current = current.next
        return current.data

# Demostración de uso
dll = DoublyLinkedList()
dll.append_right(10)
dll.append_right(20)
dll.append_left(5)
print(f"Lista: [5, 10, 20] (simulado)")
print(f"Pop left: {dll.pop_left()} -> queda [10, 20]")
print(f"Pop right: {dll.pop_right()} -> queda [10]")
print(f"Elemento en índice 0: {dll.get_at_index(0)}")

In [ ]:
# Medición de tiempos
def time_linked_list_ops(n=1000):
    dll = DoublyLinkedList()
    
    start = time.time()
    for i in range(n):
        dll.append_left(i)
    t_append_left = time.time() - start
    
    start = time.time()
    for i in range(n):
        dll.append_right(i)
    t_append_right = time.time() - start
    
    start = time.time()
    for i in range(n):
        dll.get_at_index(i)
    t_get = time.time() - start
    
    print(f"Tiempo para {n} append_left: {t_append_left:.6f} s (O(1) esperado)")
    print(f"Tiempo para {n} append_right: {t_append_right:.6f} s (O(1) esperado)")
    print(f"Tiempo para {n} get_at_index: {t_get:.6f} s (O(n) esperado)")

time_linked_list_ops(1000)

### 3.2. Conexión con motores

*   **Redis Lists**: Implementadas como listas doblemente enlazadas, permiten operaciones `LPUSH`, `RPUSH`, `LPOP`, `RPOP` en tiempo constante. Ideales para colas y pilas.
*   **Modelo de cola**: Una cola FIFO se implementa con `LPUSH` + `RPOP` (o viceversa).

---
## Actividad 4: Skip Lists - La Ingeniería Detrás de los Conjuntos Ordenados

Las Skip Lists combinan capas de listas enlazadas para lograr búsqueda O(log n) promedio.

### 4.1. Visualización interactiva

**Instrucciones:**
1. Abre el [Skip List Visualizer de la USFCA](https://www.cs.usfca.edu/~galles/visualization/SkipList.html).
2. Inserta algunos números (por ejemplo, 10, 20, 5, 15, 25).
3. Observa las múltiples capas y cómo la búsqueda salta niveles.
4. Realiza una búsqueda y observa el camino.

### 4.2. Conexión con motores

*   **Redis Sorted Sets (ZSET)**: Implementados con Skip Lists, permiten operaciones como `ZRANGE`, `ZRANK`, `ZSCORE` con eficiencia logarítmica.

In [ ]:
# Simulación conceptual: búsqueda en Skip List vs lista enlazada simple
import math

n = 1_000_000
print(f"Para {n:,} elementos:")
print(f"  Lista enlazada simple: búsqueda O(n) ~ {n/1e6:.1f} millones de pasos")
print(f"  Skip List: búsqueda O(log n) ~ {math.log2(n):.1f} pasos")

---
## Actividad 5: LSM-Trees - La Base de las Bases de Datos de Alto Rendimiento en Escritura

LSM-Trees (Log-Structured Merge-Trees) optimizan escrituras secuenciales y son la base de Cassandra, HBase, RocksDB, LevelDB.

### 5.1. Simulación conceptual de un LSM-Tree

In [ ]:
import bisect

class LSMTreeSim:
    """Simulación muy simplificada de un LSM-Tree."""
    def __init__(self, memtable_size=5):
        self.memtable = {}  # escrituras recientes (en memoria)
        self.memtable_size = memtable_size
        self.sstables = []  # lista de SSTables (archivos ordenados en disco)
        self.bloom_filters = []  # bloom filters simulados (aquí solo un set)
    
    def insert(self, key, value):
        self.memtable[key] = value
        if len(self.memtable) >= self.memtable_size:
            self._flush()
    
    def _flush(self):
        """Descarga memtable a SSTable (ordenado)."""
        # Ordenar por clave
        sorted_items = sorted(self.memtable.items())
        self.sstables.append(sorted_items)
        # Simular bloom filter: guardamos un set de claves presentes
        self.bloom_filters.append(set(self.memtable.keys()))
        self.memtable = {}
        print(f"  Flushed: nueva SSTable con {len(sorted_items)} registros. Total SSTables: {len(self.sstables)}")
    
    def search(self, key):
        # 1. Buscar en memtable
        if key in self.memtable:
            return self.memtable[key]
        
        # 2. Buscar en SSTables (de más reciente a más antigua)
        for i, (sstable, bf) in enumerate(zip(reversed(self.sstables), reversed(self.bloom_filters))):
            # Simular bloom filter: si el key no está en el set, pasamos
            if key not in bf:
                continue
            # Búsqueda binaria en la SSTable (porque está ordenada)
            keys = [k for k, _ in sstable]
            pos = bisect.bisect_left(keys, key)
            if pos < len(keys) and keys[pos] == key:
                return sstable[pos][1]
        return None

# Simulación
lsm = LSMTreeSim(memtable_size=4)

for i in range(20):
    lsm.insert(f"key_{i}", f"val_{i}")

print("\nBúsquedas:")
print(f"key_5: {lsm.search('key_5')}")
print(f"key_15: {lsm.search('key_15')}")
print(f"key_99: {lsm.search('key_99')}")

### 5.2. Explicación de Bloom Filters

Los Bloom Filters son estructuras probabilísticas que permiten determinar si un elemento **no está** en un conjunto (con certeza) o **puede estar** (con cierta probabilidad de falso positivo). En LSM-Trees, evitan buscar en SSTables donde seguro no está el dato, reduciendo lecturas innecesarias.

---
## Actividad 6: Índices Invertidos - El Corazón de la Búsqueda de Texto

Los índices invertidos mapean términos a documentos donde aparecen.

### 6.1. Construcción de un índice invertido simple

In [ ]:
documents = [
    "el gato duerme en el sofá",
    "el perro juega en el jardín",
    "el gato y el perro son amigos"
]

def build_inverted_index(docs):
    inverted = defaultdict(list)
    for doc_id, doc in enumerate(docs):
        words = doc.lower().split()
        for word in set(words):  # usar set para evitar duplicados por documento
            inverted[word].append(doc_id)
    return inverted

inverted_idx = build_inverted_index(documents)

print("Índice invertido:")
for term, doc_ids in sorted(inverted_idx.items()):
    print(f"  '{term}': {doc_ids}")

def search(term):
    term = term.lower()
    if term in inverted_idx:
        return inverted_idx[term]
    return []

print("\nBúsqueda de 'gato':", search("gato"))
print("Búsqueda de 'perro':", search("perro"))

### 6.2. Conexión con motores

*   **Elasticsearch y Solr** (basados en Lucene) utilizan índices invertidos para búsqueda de texto completo.
*   **MongoDB** ofrece índices de tipo `text` para búsqueda de texto.

---
## Actividad 7: Estructuras de Datos en Grafos - Nodos y Relaciones

Los grafos representan entidades (nodos) y sus conexiones (aristas).

### 7.1. Representación de un grafo en Python (lista de adyacencia)

In [ ]:
class Graph:
    def __init__(self):
        self.adjacency = defaultdict(list)
    
    def add_edge(self, u, v):
        self.adjacency[u].append(v)
        self.adjacency[v].append(u)  # no dirigido
    
    def get_neighbors(self, u):
        return self.adjacency.get(u, [])
    
    def dfs_limited(self, start, depth_limit):
        visited = set()
        result = []
        
        def _dfs(node, depth):
            if depth > depth_limit:
                return
            if node not in visited:
                visited.add(node)
                result.append(node)
                for neighbor in self.adjacency.get(node, []):
                    _dfs(neighbor, depth+1)
        
        _dfs(start, 0)
        return result

g = Graph()
edges = [('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','F'), ('E','F')]
for u, v in edges:
    g.add_edge(u, v)

print("Vecinos de 'A':", g.get_neighbors('A'))
print("DFS desde 'A' con profundidad 2:", g.dfs_limited('A', 2))

### 7.2. Conexión con motores

*   **Neo4j** está optimizado para almacenar nodos y relaciones como estructuras de primer nivel, permitiendo recorridos de grafos (traversals) sin costosos JOINs.

---
## Actividad de Cierre: Mapeo Motor-Estructura

Completa la siguiente tabla resumen, relacionando cada motor de base de datos con su estructura de datos principal y la complejidad de sus operaciones clave.

In [ ]:
# Crear DataFrame vacío para la tabla resumen
columns = ['Motor', 'Estructura Principal', 'Operación Clave', 'Complejidad']
data = [
    ['PostgreSQL (índice por defecto)', 'B-Tree', 'Búsqueda por igualdad/rango', 'O(log n)'],
    ['MySQL (InnoDB)', 'B-Tree', 'Búsqueda por igualdad/rango', 'O(log n)'],
    ['MongoDB (índice por defecto)', 'B-Tree', 'Búsqueda por igualdad/rango', 'O(log n)'],
    ['Redis (Strings)', 'Tabla Hash', 'GET/SET', 'O(1)'],
    ['Redis (Lists)', 'Lista doblemente enlazada', 'LPUSH/RPOP', 'O(1)'],
    ['Redis (Sorted Sets)', 'Skip List', 'ZRANGE/ZSCORE', 'O(log n)'],
    ['Cassandra', 'LSM-Tree', 'Escritura secuencial', 'O(log n) escritura, O(log n) lectura con bloom'],
    ['Elasticsearch', 'Índice Invertido', 'Búsqueda de texto', 'O(1) por término'],
    ['Neo4j', 'Grafo (lista de adyacencia)', 'Recorrido de vecinos', 'O(1) por arista'],
]

df_summary = pd.DataFrame(data, columns=columns)
df_summary

**Instrucciones para el estudiante:** Investiga y completa la tabla con al menos 3 motores adicionales (por ejemplo, SQLite, HBase, ArangoDB). Para cada uno, identifica la estructura principal y la complejidad de sus operaciones.

---
## Ejercicios para el Estudiante

1.  **B-Tree:** Usa el visualizador web con un orden 3 y explica qué ocurre cuando insertas claves en orden ascendente (1,2,3,4,5,...). ¿El árbol se mantiene balanceado?

2.  **Tabla Hash:** Modifica la implementación de la tabla hash para que se redimensione automáticamente cuando el factor de carga supere 0.7. Mide cómo mejoran los tiempos.

3.  **Lista Enlazada vs Lista de Python:** Compara el rendimiento de nuestra lista doblemente enlazada con la lista nativa de Python (`list`) para inserciones al inicio. ¿Por qué la lista nativa es más lenta en `insert(0, valor)`?

4.  **LSM-Tree:** En la simulación, añade un contador de accesos a SSTables. ¿Cómo afecta el bloom filter al número de SSTables revisadas?

5.  **Índice Invertido:** Modifica el índice para almacenar también la frecuencia del término en cada documento. ¿Cómo ayudaría esto a ranking de relevancia?

6.  **Reflexión:** ¿Por qué crees que Cassandra (LSM-Tree) es mejor para escrituras intensivas, mientras que PostgreSQL (B-Tree) es mejor para lecturas consistentes?

---
## Conclusiones de la Sesión

*   Las bases de datos son implementaciones de **estructuras de datos clásicas** optimizadas para persistencia y concurrencia.
*   Los **B-Trees** son la base de los índices SQL, ofreciendo búsqueda O(log n).
*   Las **tablas hash** permiten acceso O(1) y son la base de sistemas clave-valor como Redis.
*   Las **listas enlazadas** (en Redis) implementan colas y pilas eficientes.
*   Las **Skip Lists** (en Redis Sorted Sets) combinan listas enlazadas con múltiples capas para búsqueda logarítmica.
*   Los **LSM-Trees** optimizan escrituras y son la base de Cassandra, HBase, etc.
*   Los **índices invertidos** (Elasticsearch, Solr) permiten búsqueda de texto rápido.
*   Los **grafos** (Neo4j) modelan relaciones de forma natural con recorridos eficientes.

¡Ahora comprendes el "backend" de los motores de bases de datos!